See this issue: https://github.com/opera-adt/burst_db/issues/120

Simply download the geopackage data and update it. The geometries are more complex so the file is larger...

In [1]:
import geopandas as gpd
import pandas as pd
from shapely import Polygon, MultiPolygon, LineString

In [2]:
df_burst_gpkg = gpd.read_file('opera-s1-disp-0.12.0.gpkg')
df_burst_gpkg.head()

/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/pyogrio/geopandas.py:275: UserWarning: More than one layer found in 'opera-s1-disp-0.12.0.gpkg': 'burst_id_map' (default), 'frames', 'frames_bursts', 'metadata'. Specify layer parameter to avoid this warning.
  result = read_func(


,burst_id,subswath_name,relative_orbit_number,time_from_anx_sec,orbit_pass,burst_id_jpl,epsg,is_north_america,xmin,ymin,xmax,ymax,geometry
0,1,IW1,1,2.301015,ASCENDING,t001_000001_iw1,32631,False,531360,78240,630690,128280,"MULTIPOLYGON Z (((3.36758 0.75322 0, 3.763 0.8..."
1,1,IW2,1,3.133218,ASCENDING,t001_000001_iw2,32631,False,612180,102660,715320,152370,"MULTIPOLYGON Z (((4.09178 0.97402 0, 4.50253 1..."
2,1,IW3,1,4.211255,ASCENDING,t001_000001_iw3,32631,False,696870,126810,789960,175590,"MULTIPOLYGON Z (((4.85545 1.192 0, 5.21485 1.2..."
3,2,IW1,1,5.059288,ASCENDING,t001_000002_iw1,32631,False,527460,96690,626820,146730,"MULTIPOLYGON Z (((3.33269 0.92005 0, 3.7281 1...."
4,2,IW2,1,5.891491,ASCENDING,t001_000002_iw2,32631,False,608280,121080,711450,170760,"MULTIPOLYGON Z (((4.0569 1.14071 0, 4.46766 1...."


In [3]:
df_burst_enum = gpd.read_parquet('jpl_burst_geo_old.parquet')
df_burst_enum.head()

,jpl_burst_id,geometry
0,T001-000025-IW3,"POLYGON ((4.02659 5.18854, 4.73485 5.32978, 4...."
1,T001-000026-IW2,"POLYGON ((3.22392 5.14115, 4.02565 5.30297, 3...."
2,T001-000026-IW3,"POLYGON ((3.99185 5.35497, 4.70037 5.49605, 4...."
3,T001-000027-IW1,"POLYGON ((2.46164 5.09048, 3.22658 5.24523, 3...."
4,T001-000027-IW2,"POLYGON ((3.18963 5.30788, 3.99158 5.46953, 3...."


In [4]:
df_gpkg_updated = df_burst_gpkg[['burst_id_jpl', 'geometry']].copy()

In [5]:
df_gpkg_updated['jpl_burst_id'] = df_gpkg_updated['burst_id_jpl'].str.replace('_', '-').str.upper()
df_gpkg_updated = df_gpkg_updated[['jpl_burst_id', 'geometry']].copy()
df_gpkg_updated['geometry'] = df_gpkg_updated['geometry'].force_2d()

In [6]:
df_gpkg_updated.head()

,jpl_burst_id,geometry
0,T001-000001-IW1,"MULTIPOLYGON (((3.36758 0.75322, 3.763 0.83555..."
1,T001-000001-IW2,"MULTIPOLYGON (((4.09178 0.97402, 4.50253 1.059..."
2,T001-000001-IW3,"MULTIPOLYGON (((4.85545 1.192, 5.21485 1.26631..."
3,T001-000002-IW1,"MULTIPOLYGON (((3.33269 0.92005, 3.7281 1.0023..."
4,T001-000002-IW2,"MULTIPOLYGON (((4.0569 1.14071, 4.46766 1.2262..."


In [7]:
df_final = pd.merge(df_burst_enum[['jpl_burst_id']], df_gpkg_updated, on='jpl_burst_id', how='left')

In [8]:
df_final.head()

,jpl_burst_id,geometry
0,T001-000025-IW3,"MULTIPOLYGON (((4.02659 5.18854, 4.38755 5.260..."
1,T001-000026-IW2,"MULTIPOLYGON (((3.22392 5.14115, 3.63636 5.224..."
2,T001-000026-IW3,"MULTIPOLYGON (((3.99185 5.35497, 4.35294 5.427..."
3,T001-000027-IW1,"MULTIPOLYGON (((2.46164 5.09048, 2.8585 5.1709..."
4,T001-000027-IW2,"MULTIPOLYGON (((3.18963 5.30788, 3.60218 5.391..."


Sanity check

In [9]:
df_final.geometry.map(lambda geo: geo.is_empty).sum()

np.int64(0)

In [10]:
def update_geometry_low_tech(geo):
    if isinstance(geo, Polygon):
        return geo
    elif isinstance(geo, MultiPolygon):
        geoms = list(geo.geoms)
        if len(geoms) == 1:
            geo = geoms[0]
        return geo
    else:
        raise ValueError('weird geometry!')

In [11]:
df_final_burst = gpd.GeoDataFrame(df_final, geometry=df_final.geometry, crs='4326')
df_final_burst['geometry'] = df_final_burst['geometry'].map(update_geometry_low_tech)

In [12]:
df_final_burst.head()

,jpl_burst_id,geometry
0,T001-000025-IW3,"POLYGON ((4.02659 5.18854, 4.38755 5.26068, 4...."
1,T001-000026-IW2,"POLYGON ((3.22392 5.14115, 3.63636 5.2246, 4.0..."
2,T001-000026-IW3,"POLYGON ((3.99185 5.35497, 4.35294 5.42703, 4...."
3,T001-000027-IW1,"POLYGON ((2.46164 5.09048, 2.8585 5.17098, 3.2..."
4,T001-000027-IW2,"POLYGON ((3.18963 5.30788, 3.60218 5.39124, 3...."


In [13]:
df_final_burst.to_parquet('jpl_burst_geo.parquet', compression='zstd')

In [14]:
df_final_burst.geometry[0].wkt

'POLYGON ((4.026586 5.188537, 4.387546 5.260678, 4.734847 5.329782, 4.692665 5.533879, 4.346147 5.460323, 3.985968 5.38373, 4.026586 5.188537))'

In [15]:
df_burst_enum.geometry[0].wkt

'POLYGON ((4.026586 5.188537, 4.734847 5.329782, 4.692665 5.533879, 3.985968 5.38373, 4.026586 5.188537))'

In [16]:
antimeridian_0 = LineString(coordinates=((-180, 90), (-180, -90)))#.buffer(0.00000001)
antimeridian_1 = LineString(coordinates=((180, 90), (180, -90)))#.buffer(0.00000001)

for antimeridian in [antimeridian_0, antimeridian_1]:
    ind_anti = df_final_burst.geometry.intersects(antimeridian)
    df_final_burst_antimerid = df_final_burst[ind_anti].reset_index(drop=True)

    any_polygons = (df_final_burst_antimerid.geometry.map(lambda geo: isinstance(geo, Polygon))).sum()
    assert any_polygons == 0

    df_mgrs_antimerid_not = df_final_burst[~ind_anti].reset_index(drop=True)
    any_multis = (df_mgrs_antimerid_not.geometry.map(lambda geo: isinstance(geo, MultiPolygon))).sum()
    assert any_multis == 0